# Materia: Procesamiento del habla.
# Alumno: Moises Lobayza

# Análisis de similitud semántica con spaCy

En este trabajo se analiza la similitud semántica entre pares de palabras en español utilizando embeddings de spaCy y la similitud del coseno.

El objetivo es encontrar tres tipos de pares:

- Un par con similitud positiva alta, cercana a 1.
- Un par con similitud neutral, cercana a 0.
- Un par con similitud negativa, o el valor más negativo encontrado.


***Nota: el notebook de ejemplo de clase usa Gensim con GloVe como introducción a embeddings. En esta entrega se usa spaCy porque la consigna pide explícitamente embeddings en español de spaCy.***

## 1. Instalación de paquetes


Usaremos `spaCy`, `numpy` y `pandas`.  
El modelo elegido es `es_core_news_md`, porque incluye vectores útiles para calcular similitud semántica.

In [ ]:
!pip install -U spacy pandas numpy
!python -m spacy download es_core_news_md


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 12.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 2. Cargar librerías y modelo



In [ ]:
import spacy
import numpy as np
import pandas as pd
from itertools import combinations

nlp = spacy.load("es_core_news_md")

print("Modelo cargado:", nlp.meta["name"])
print("Idioma:", nlp.meta["lang"])


Modelo cargado: core_news_md
Idioma: es


## 3. Similitud del coseno

La similitud del coseno permite comparar dos vectores según el ángulo entre ellos.

$\cos\theta = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$

Su valor puede ir de -1 a 1:

- Cercano a **1**: vectores muy similares.
- Cercano a **0**: vectores sin relación clara.
- Cercano a **-1**: vectores en direcciones opuestas.

En embeddings de palabras no siempre aparecen valores negativos fuertes. Por eso la consigna permite usar el valor más negativo que se logre encontrar.

## 4. Funciones para obtener vectores y calcular similitud

In [ ]:
def obtener_vector(palabra):
    """
    Procesa una palabra con spaCy y devuelve su vector.
    """
    doc = nlp(palabra)

    if not doc.has_vector or doc.vector_norm == 0:
        raise ValueError(f"La palabra '{palabra}' no tiene vector válido.")

    return doc.vector


def cosine_similarity(vec1, vec2):
    """
    Calcula la similitud del coseno entre dos vectores.
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def similitud_coseno(palabra_1, palabra_2):
    """
    Calcula la similitud del coseno entre dos palabras.
    """
    vec1 = obtener_vector(palabra_1)
    vec2 = obtener_vector(palabra_2)
    return cosine_similarity(vec1, vec2)


## 5. Lista de palabras candidatas

Se arma una lista con palabras de distintos campos semánticos para que el código pueda encontrar pares muy similares, pares neutros y pares con baja similitud.


In [ ]:
palabras = [
    "médico", "doctor", "hospital", "enfermera", "salud", "enfermedad", "paciente",
    "coche", "auto", "automóvil", "camión", "bicicleta", "avión", "tren",
    "perro", "gato", "caballo", "pez", "león", "tigre", "vaca",
    "rey", "reina", "hombre", "mujer", "niño", "adulto",
    "feliz", "contento", "alegre", "triste", "amor", "odio",
    "computadora", "ordenador", "teclado", "software", "internet", "programa",
    "economía", "mercado", "dinero", "banco", "inflación", "precio",
    "montaña", "río", "playa", "bosque", "desierto", "océano",
    "cuchara", "mesa", "zapato", "almohada", "triángulo", "democracia",
    "fútbol", "boxeo", "música", "pintura", "poesía", "cine"
]


## 6. Calcular similitud entre todos los pares posibles


In [ ]:
resultados = []

for palabra_1, palabra_2 in combinations(palabras, 2):
    try:
        sim = similitud_coseno(palabra_1, palabra_2)
        resultados.append({
            "palabra_1": palabra_1,
            "palabra_2": palabra_2,
            "similitud_coseno": sim
        })
    except ValueError as error:
        print(error)

df = pd.DataFrame(resultados)

df.head()


,palabra_1,palabra_2,similitud_coseno
0,médico,doctor,0.637438
1,médico,hospital,0.603883
2,médico,enfermera,0.648539
3,médico,salud,0.432393
4,médico,enfermedad,0.437778


## 7. Buscar pares positivos, neutros y negativos


In [ ]:
# Par con mayor similitud positiva
positivos = df.sort_values("similitud_coseno", ascending=False)

# Par con similitud más cercana a 0
neutrales = df.copy()
neutrales["distancia_a_cero"] = neutrales["similitud_coseno"].abs()
neutrales = neutrales.sort_values("distancia_a_cero")

# Par con menor similitud encontrada
negativos = df.sort_values("similitud_coseno", ascending=True)

print("Pares con similitud positiva alta:")
display(positivos.head(10))

print("Pares con similitud cercana a 0:")
display(neutrales.head(10))

print("Pares con menor similitud encontrada:")
display(negativos.head(10))


Pares con similitud positiva alta:


,palabra_1,palabra_2,similitud_coseno
777,perro,gato,0.848729
1518,computadora,ordenador,0.777351
963,león,tigre,0.761811
468,auto,automóvil,0.745505
414,coche,automóvil,0.737408
1680,economía,inflación,0.732674
1250,niño,adulto,0.708877
413,coche,auto,0.684726
522,automóvil,camión,0.682425
415,coche,camión,0.677247


Pares con similitud cercana a 0:


,palabra_1,palabra_2,similitud_coseno,distancia_a_cero
1328,feliz,computadora,0.000071,0.000071
341,enfermedad,playa,0.000273,0.000273
1613,software,bosque,0.000300,0.000300
1061,vaca,odio,0.000507,0.000507
1048,tigre,poesía,-0.000649,0.000649
1306,adulto,río,-0.000730,0.000730
1227,mujer,mercado,0.000876,0.000876
1138,reina,contento,0.000888,0.000888
1760,banco,pintura,-0.001005,0.001005
173,hospital,zapato,-0.001063,0.001063


Pares con menor similitud encontrada:


,palabra_1,palabra_2,similitud_coseno
1432,triste,programa,-0.226701
1434,triste,mercado,-0.212683
1430,triste,software,-0.210180
1146,reina,software,-0.207152
1025,tigre,programa,-0.186858
1668,programa,almohada,-0.169844
1761,banco,poesía,-0.164283
1067,vaca,programa,-0.162460
852,gato,inflación,-0.161581
1661,programa,playa,-0.148590


## 8. Selección final de los tres pares


In [ ]:
pares_finales = pd.DataFrame([
    {
        "tipo": "Similitud positiva",
        "palabra_1": positivos.iloc[0]["palabra_1"],
        "palabra_2": positivos.iloc[0]["palabra_2"],
        "similitud_coseno": positivos.iloc[0]["similitud_coseno"]
    },
    {
        "tipo": "Similitud neutral",
        "palabra_1": neutrales.iloc[0]["palabra_1"],
        "palabra_2": neutrales.iloc[0]["palabra_2"],
        "similitud_coseno": neutrales.iloc[0]["similitud_coseno"]
    },
    {
        "tipo": "Similitud negativa",
        "palabra_1": negativos.iloc[0]["palabra_1"],
        "palabra_2": negativos.iloc[0]["palabra_2"],
        "similitud_coseno": negativos.iloc[0]["similitud_coseno"]
    }
])

pares_finales


,tipo,palabra_1,palabra_2,similitud_coseno
0,Similitud positiva,perro,gato,0.848729
1,Similitud neutral,feliz,computadora,0.000071
2,Similitud negativa,triste,programa,-0.226701


## 9. Interpretación de resultados

### Similitud positiva

El par con similitud positiva alta fue **[perro]** y **[gato]**, con un valor aproximado de **[0.848729]**.

Este resultado tiene sentido porque ambas palabras pertenecen a un campo semántico similar. Es decir, suelen utilizarse en contextos parecidos dentro del lenguaje. Por eso, sus vectores apuntan en direcciones cercanas dentro del espacio de embeddings.

### Similitud neutral

El par con similitud más cercana a 0 fue **[feliz]** y **[computadora]**, con un valor aproximado de **[0.000071]**.

Esto indica que las palabras no tienen una relación semántica fuerte según el modelo. Probablemente pertenecen a campos de significado diferentes y no suelen aparecer en contextos lingüísticos parecidos.

### Similitud negativa

El par con menor similitud encontrada fue **[triste]** y **[programa]**, con un valor aproximado de **[-0.226701]**.

En este caso, el valor negativo no debe interpretarse necesariamente como oposición directa o antonimia. En los embeddings, dos palabras opuestas pueden aparecer en contextos similares y tener una similitud relativamente alta. El valor bajo o negativo indica que, dentro del espacio vectorial del modelo, esas palabras quedaron representadas en direcciones más alejadas.

### Conclusión general

Este trabajo muestra que los embeddings permiten representar palabras como vectores numéricos. La similitud del coseno permite medir qué tan parecidas son esas representaciones. Cuando dos palabras tienen significado o uso parecido, la similitud tiende a ser alta. Cuando no tienen una relación clara, la similitud se acerca a 0. Los valores negativos son menos frecuentes y representan una separación fuerte dentro del espacio vectorial, aunque no siempre equivalen a palabras opuestas.


## 10. Fuentes utilizadas

- Documentación oficial de spaCy sobre modelos y pipelines.
- Documentación oficial de spaCy sobre vectores de palabras y similitud semántica.
- Notebook de clase sobre embeddings.

